# 03 — Get real-world data

Here we use a real dataset. **Forest Covertype** from the UCI Machine Learning
Repository describes plots of forest land in northern Colorado. Each plot has
measurements such as elevation, slope, distance to water and roads, hillshade, and which
wilderness area and soil type it sits on, plus labelled with the tree species that dominates it.

This notebook downloads it, unpacks it.

Dataset: <https://archive.ics.uci.edu/dataset/31/covertype>

*Reuse of this database is unlimited with retention of the copyright notice for
Jock A. Blackard and Colorado State University.*

In [ ]:
import zipfile
from pathlib import Path
from urllib.request import urlretrieve

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

URL = "https://archive.ics.uci.edu/static/public/31/covertype.zip"
DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

RANDOM_STATE = 0
SAMPLE_SIZE = 20_000

### The columns

The file has **no header row**; the names live in the dataset documentation. There are 54
of them:

* 10 numeric measurements (elevation, aspect, slope, distances, hillshade),
* 4 binary `Wilderness_Area` flags,
* 40 binary `Soil_Type` flags.

Column 55 is the label: the forest cover type, 1 to 7.

In [ ]:
FEATURES = (
    ["Elevation", "Aspect", "Slope",
     "Horizontal_Distance_To_Hydrology", "Vertical_Distance_To_Hydrology",
     "Horizontal_Distance_To_Roadways",
     "Hillshade_9am", "Hillshade_Noon", "Hillshade_3pm",
     "Horizontal_Distance_To_Fire_Points"]
    + [f"Wilderness_Area_{i}" for i in range(1, 5)]
    + [f"Soil_Type_{i}" for i in range(1, 41)]
)
LABEL = "Cover_Type"

CLASS_NAMES = {1: "Spruce/Fir", 2: "Lodgepole Pine", 3: "Ponderosa Pine",
               4: "Cottonwood/Willow", 5: "Aspen", 6: "Douglas-fir",
               7: "Krummholz"}

print(f"{len(FEATURES)} feature columns + 1 label column")

### Download and unpack

The archive holds a single gzipped CSV (`covtype.data.gz`). pandas reads gzip directly.

The download is skipped if the file is already there.

In [ ]:
zip_path = DATA_DIR / "covertype.zip"
gz_path = DATA_DIR / "covtype.data.gz"

if not gz_path.exists():
    if not zip_path.exists():
        print("downloading ...")
        urlretrieve(URL, zip_path)
    with zipfile.ZipFile(zip_path) as z:
        print("archive contains:", z.namelist())
        z.extract("covtype.data.gz", DATA_DIR)

print(f"\n{gz_path}  ({gz_path.stat().st_size / 1e6:.1f} MB compressed)")

### Read it

In [ ]:
df = pd.read_csv(gz_path, header=None, names=FEATURES + [LABEL])

print(f"{df.shape[0]:,} rows x {df.shape[1]} columns")
print(f"{df.memory_usage(deep=True).sum() / 1e6:.0f} MB in memory")
df.head()

### The label

Seven species, and they are **not** equally common. One class has 283,301 examples, another
has 2,747 -> unbalance.

In [ ]:
counts = df[LABEL].value_counts().sort_index()

label_table = pd.DataFrame({
    "species": [CLASS_NAMES[i] for i in counts.index],
    "count": counts,
    "percent": (counts / len(df) * 100).round(2),
})
label_table

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.bar([CLASS_NAMES[i] for i in counts.index], counts.values, color="0.5")
ax.set_ylabel("plots")
ax.set_title("Cover type in the full dataset")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

### Keep a sample

In [ ]:
sample = df.sample(n=SAMPLE_SIZE, random_state=RANDOM_STATE)

X = sample[FEATURES].to_numpy(dtype=np.float32)
y = sample[LABEL].to_numpy(dtype=np.int8)

out = DATA_DIR / "covertype_sample.npz"
np.savez_compressed(out, X=X, y=y, feature_names=np.array(FEATURES))

print(f"X {X.shape} {X.dtype}")
print(f"y {y.shape} {y.dtype}")
print(f"saved {out}  ({out.stat().st_size / 1e6:.1f} MB)")

### The sample keeps the imbalance

A plain random sample has the same class proportions as the full data; including the rare
species.

In [ ]:
sample_counts = pd.Series(y).value_counts().sort_index()

pd.DataFrame({
    "species": [CLASS_NAMES[i] for i in sample_counts.index],
    "in full data": [int(counts[i]) for i in sample_counts.index],
    "in sample": sample_counts.values,
    "percent of sample": (sample_counts.values / SAMPLE_SIZE * 100).round(2),
})